In [1]:
import os
import pickle
from torch.utils import data
import numpy as np
import torch
import torch.utils.data as data
from IPython.display import clear_output

# Path to Shakespeare metadata
data_dir = './shakespeare_char/'

# Load the metadata dictionary
meta_path = os.path.join(data_dir, 'meta.pkl')
vocab_size = None
with open(meta_path, 'rb') as f:
    meta = pickle.load(f)

# Character and index mappings
itos = meta['itos'] # index to string (character)
stoi = meta['stoi'] # string (character) to index
print(f"itos:{itos}")
# Display the vocabulary
print(f"vocabulary: {repr(''.join(stoi.keys()))}")

# Total number of unique characters
vocab_size = meta['vocab_size']
print(f'vocabulary size: {vocab_size}')


class ShakespeareDataset(data.Dataset):
    """
    Memory-mapped dataset for character-level sequences.

    Each item is a 1D tensor of indices (torch.long) of length `context_len`
    from a rolling window over the encoded Shakespeare corpus.

    Notes
    -----
    - Uses np.memmap to avoid loading the entire file into RAM.
    - Returns only `x` (the context window).
      This will serve as the clean target for denoising.
      Noising will be applied on-the-fly during the training.
    """
    def __init__(
        self,
        data_dir: str,
        split: str = "train",
        context_len: int = 256,
        dtype: np.dtype = np.uint16,
    ) -> None:
        if split not in {"train", "val"}:
            raise ValueError(f"split must be 'train' or 'val', got: {split!r}")
        if context_len <= 0:
            raise ValueError(f"context_len must be positive, got: {context_len}")

        self.split = split
        self.context_len = int(context_len)

        bin_path = os.path.join(data_dir, f"{split}.bin")
        if not os.path.isfile(bin_path):
            raise FileNotFoundError(f"Could not find {bin_path}")

        # Memory-map the encoded corpus. uint16 matches the preprocessing.
        self.data = np.memmap(bin_path, dtype=dtype, mode="r")

        # Number of valid starting positions for a full context window
        self._n = max(0, len(self.data) - self.context_len)

    def __len__(self) -> int:
        return self._n

    def __getitem__(self, index: int) -> torch.Tensor:
        if index < 0 or index >= self._n:
            raise IndexError(f"Index {index} out of range for dataset of length {self._n}.")
        # Slice a contiguous window and convert to torch.long (int64)
        x_np = self.data[index : index + self.context_len].astype(np.int64)
        x = torch.from_numpy(x_np)  # shape: [context_len], dtype: torch.long
        return x

def get_data_loader(data_dir, split, batch_size, context_len=256):
    dataset = ShakespeareDataset(data_dir, split, context_len)
    return data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialise
batch_size = 64
context_length = 256

train_dataloader = get_data_loader(data_dir, 'train', batch_size, context_length)
val_dataloader   = get_data_loader(data_dir, 'val', batch_size, context_length)

# Peek at one batch to confirm shapes/types
batch = next(iter(train_dataloader))
print(batch.shape)

def decode(indices_tensor: torch.Tensor):
    '''Decodes a 1D tensor of indices to text'''
    indices = indices_tensor.cpu().numpy()
    return ''.join([itos[i] for i in indices])

# Check what the model is "seeing"
print(decode(batch[0]))

def perturb_batch(batch: torch.Tensor, sigma_bar: torch.Tensor) -> torch.Tensor:
    """
    Diffuse each token independently according to Eq. (3).

      - With probability e^{-sigma_bar} + (1 - e^{-sigma_bar})/N, a token stays the same.
      - Otherwise, it jumps uniformly to one of the other N-1 tokens.
    Args:
        batch: LongTensor of shape [B, L], each entry in [0, vocab_size-1]
        sigma_bar: scalar tensor
    Returns:
        batch_pert: perturbed batch of LongTensor
    """
    B, L = batch.shape

    # 1) Compute move probability: (1 - e^{-sigma}) * (1 - 1/N)
    stay_base = torch.exp(-sigma_bar)
    move_prob = (1 - stay_base) * (1 - 1 / vocab_size)

    # 2) Bernoulli: should this token move?
    move_mask = torch.rand(B, L, device=batch.device) < move_prob

    # 3) For tokens that move, sample a *different* id uniformly from the other N-1 ids.
    #    Sample r in [0, N-2], then map to [0..N-1]\{orig} by skipping the original.
    r = torch.randint(low=0, high=vocab_size - 1, size=(B, L), device=batch.device)
    # shift up by 1 wherever r >= original id, covering {0, .., k-1, k+1, .., N-1}
    new_ids = r + (r >= batch)

    # 4) Apply moves; else keep original
    batch_pert = torch.where(move_mask, new_ids, batch)
    return batch_pert

import textwrap

def print_wrapped(long_text, width=80, **kwargs):
    """
    Print text wrapped to a maximum line width, preserving paragraph breaks.
    """
    paragraphs = long_text.splitlines()
    wrapped = [textwrap.fill(p, width=width) if p else '' for p in paragraphs]
    final_text = "\n".join(wrapped)
    print(final_text, **kwargs)
    


itos:{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h', 47: 'i', 48: 'j', 49: 'k', 50: 'l', 51: 'm', 52: 'n', 53: 'o', 54: 'p', 55: 'q', 56: 'r', 57: 's', 58: 't', 59: 'u', 60: 'v', 61: 'w', 62: 'x', 63: 'y', 64: 'z'}
vocabulary: "\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
vocabulary size: 65
torch.Size([64, 256])
things long past:
Though Richard my life's counsel would not hear,
My death's sad tale may yet undeaf his ear.

DUKE OF YORK:
No; it is stopp'd with other flattering sounds,
As praises, of whose taste the wise are fond,
Lascivious metres, to whose venom so


In [2]:
import math
import torch.nn as nn
from torch.nn import functional as F
class GeometricNoise:
    def __init__(self, sigma_min=1e-4, sigma_max=20):
        self.sigmas = 1.0 * torch.tensor([sigma_min, sigma_max])

    def rate_noise(self, t):
        return self.sigmas[0] ** (1 - t) * self.sigmas[1] ** t * (self.sigmas[1].log() - self.sigmas[0].log())

    def total_noise(self, t):
        return self.sigmas[0] ** (1 - t) * self.sigmas[1] ** t

    def __call__(self, t):
        """
        Returns:
            \bar \sigma(t) and \sigma(t)
        """
        return self.total_noise(t), self.rate_noise(t)

    
class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        #print(f"x:{x.shape}, {x}")
        x = self.c_fc(x)
        #print(f"xc_fc:{x.shape}, {x}")
        x = self.gelu(x)
        #print(f"xgelu:{x.shape}, {x}")
        x = self.c_proj(x)
        #print(f"xc_proj:{x.shape}, {x}")
        x = self.dropout(x)
        return x
    
class SelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        # flash attention make GPU go brrrrr but support is only in PyTorch >= 2.0
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        ln = self.c_attn(x)
        #print(f"ln:{B},{T},{C}:{ln}")
        q, k, v  = ln.split(self.n_embd, dim=2)
        #print(f"q:{q}\nk:{k}\nv:{v}")
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        
        # self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        if self.flash:
            # efficient attention using Flash Attention CUDA kernels
            y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=False)
        else:
            # manual implementation of attention
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        #print(f":{y.shape}")
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side
        #print(f"y:{y}")
        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y
from typing import Optional

def modulate(x: torch.Tensor, shift: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
    return x * (1 + scale) + shift

def bias_add_scale(
    x: torch.Tensor, bias: Optional[torch.Tensor], scale: torch.Tensor, residual: Optional[torch.Tensor]) -> torch.Tensor:
    if bias is not None:
        #print("herere")
        out = scale * (x + bias)
    else:
        #print(f"scale:{scale}")
        out = scale * x
        #print(f"out:{out}")

    if residual is not None:
        #print(f"out:{out.shape}, {out}")
        #print(f"residual:{residual.shape},{residual}")
        out = residual + out
        #print(f"out:{out.shape},{out}")
    return out

class DDiTBlock(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.attn = SelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

        self.adaLN_modulation = nn.Linear(config.cond_dim, 6 * config.n_embd)
        self.adaLN_modulation.weight.data.zero_()
        self.adaLN_modulation.bias.data.zero_()

    def forward(self, x, c):
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = self.adaLN_modulation(c)[:, None].chunk(6, dim=2)
        #print(f"shift_msa:{shift_msa}")
        #print(f"scale_msa:{scale_msa}")
        x_skip = x
        ln_1 = self.ln_1(x)
        #print(f"ln_1:{ln_1}")
        x = modulate(ln_1 , shift_msa, scale_msa)
        #print(f"shift_msa:{shift_msa}")
        #print(f"scale_msa:{scale_msa}")
        x = self.attn(x)
        #print(f"x:{x}")
        ln1_again = self.ln_1(x)
        #print(f"ln1_again:{ln1_again}")
        attn_again = self.attn(ln1_again)
        #print(f"attn_again:{attn_again}")
        x = bias_add_scale(attn_again, None, gate_msa, x_skip)
        #print(f"x:{x}")
        ln_2 = self.ln_2(x)
        #print(f"ln_2:{ln_2}")
        modRes = modulate(ln_2, shift_mlp, scale_mlp)
        #print(f"modRes:{modRes}")
        mlpRes = self.mlp(modRes)
        #print(f"xBefore:{x}")
        x = bias_add_scale(mlpRes, None, gate_mlp, x)
        #print(f"mlpRes:{mlpRes}")
        #print(f"gate_mlp{}:{gate_mlp}")
        #print(f"xAfter:{x}")
        return x
class DDitFinalLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm_final = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.linear = nn.Linear(config.n_embd, config.vocab_size)
        self.linear.weight.data.zero_()
        self.linear.bias.data.zero_()

        self.adaLN_modulation = nn.Linear(config.cond_dim, 2 * config.n_embd)
        self.adaLN_modulation.weight.data.zero_()
        self.adaLN_modulation.bias.data.zero_()


    def forward(self, x, c):
        shift, scale = self.adaLN_modulation(c)[:, None].chunk(2, dim=2)
        lf = self.norm_final(x)
        #print(f"lf:{lf}")
        #print(f"shift:{shift}")
        #print(f"scale:{scale}")
        x = modulate(lf, shift, scale)
        #print(f"x:{x}")
        x = self.linear(x)
        #print(f"x:{x}")
        return x
class TimestepEmbedder(nn.Module):
    """
    Embeds scalar timesteps into vector representations.
    """
    def __init__(self, hidden_size, frequency_embedding_size=256, silu=True):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(frequency_embedding_size, hidden_size, bias=True),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size, bias=True),
        )
        self.frequency_embedding_size = frequency_embedding_size

    @staticmethod
    def timestep_embedding(t, dim, max_period=10000):
        """
        Create sinusoidal timestep embeddings.
        :param t: a 1-D Tensor of N indices, one per batch element.
                          These may be fractional.
        :param dim: the dimension of the output.
        :param max_period: controls the minimum frequency of the embeddings.
        :return: an (N, D) Tensor of positional embeddings.
        """
        # https://github.com/openai/glide-text2im/blob/main/glide_text2im/nn.py
        half = dim // 2
        freqs = torch.exp(
            -math.log(max_period) * torch.arange(start=0, end=half, dtype=torch.float32) / half
        ).to(device=t.device)
        args = t[:, None].float() * freqs[None]
        embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        if dim % 2:
            embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
        return embedding

    def forward(self, t):
        t_freq = self.timestep_embedding(t, self.frequency_embedding_size)
        #print(t_freq)
        #print("Linear 0:",self.mlp[0].weight.shape)
        #print("Weights: ", self.mlp[0].weight)
        #print("Linear 0:",self.mlp[0].bias.shape)
        #print("Linear 2:",self.mlp[2].weight.shape)
        #print("Linear 2:",self.mlp[2].bias.shape)
        t_emb = self.mlp(t_freq)
        #print("Out: ",t_emb)
        return t_emb

class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config
        self.sigma_map = TimestepEmbedder(config.cond_dim)
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([DDiTBlock(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = DDitFinalLayer(config)

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %.2fM" % (self.get_num_params()/1e6,))

    def get_num_params(self, non_embedding=True):
        """
        Return the number of parameters in the model.
        For non-embedding count (default), the position embeddings get subtracted.
        The token embeddings would too, except due to the parameter sharing these
        params are actually used as weights in the final layer, so we include them.
        """
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, sigma):
        sigma = sigma.reshape(-1)
        device = idx.device
        b, t = idx.size()
        #print("Sigma:", sigma, b, t)
        c = F.silu(self.sigma_map(sigma))
        #print("After Sigma:", c)
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device) # shape (t)
        #print(f"pos: {pos}")
        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (t, n_embd)
        #print(f"xBefore: {tok_emb + pos_emb}")
        x = self.transformer.drop(tok_emb + pos_emb)
        #print(f"xAfter: {x}")
        #assert(torch.allclose(x, tok_emb + pos_emb))
        for block in self.transformer.h:
            x = block(x, c)
            #break
        #print(f"xAfter:{x}")
        x = self.transformer.ln_f(x)
        #print(f"xLn:{x}")
        # inference-time mini-optimization: only forward the lm_head on the very last position
        x = self.lm_head(x, c) # note: using list [-1] to preserve the time dim
        #print(f"x:{x}")
        #print(f"{x.shape}, {idx.shape}")
        x = torch.scatter(x, -1, idx[..., None], torch.zeros_like(x[..., :1]))
        #print(f"idx:{idx}")
        #print(f"x:{x}")
        return x


In [3]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304 # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    cond_dim: int = 64
    dropout: float = 0.0
    bias: bool = False # True: bias in Linears and LayerNorms, like GPT-2. False: a bit better and faster

# A character-level baby GPT model :)
n_layer = 6
n_head = 6
n_embd = 384
cond_dim = 64
block_size = context_length
dropout = 0.2
bias = False # do we use bias inside LayerNorm and Linear layers?

model_args = dict(n_layer=n_layer, n_head=n_head, n_embd=n_embd, cond_dim=cond_dim,
                  bias=bias, vocab_size=vocab_size, block_size=block_size, dropout=dropout)

config = GPTConfig(**model_args)
model = GPT(config)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

sigma_min, sigma_max = 1e-4, 20
noise = GeometricNoise(sigma_min=sigma_min, sigma_max=sigma_max)


number of parameters: 11.64M


/home/murage/.local/lib/python3.8/site-packages/torch/cuda/__init__.py:128: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 10010). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


In [4]:
from safetensors.torch import load_file

state_dict = load_file(
    "converted_safetensors/model_epoch_25.safetensors",
    device=str(device)  # e.g. "cpu" or "cuda:0"
)

model.load_state_dict(state_dict)
model.eval()

GPT(
  (sigma_map): TimestepEmbedder(
    (mlp): Sequential(
      (0): Linear(in_features=256, out_features=64, bias=True)
      (1): SiLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (transformer): ModuleDict(
    (wte): Embedding(65, 384)
    (wpe): Embedding(256, 384)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x DDiTBlock(
        (ln_1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (attn): SelfAttention(
          (c_attn): Linear(in_features=384, out_features=1152, bias=False)
          (c_proj): Linear(in_features=384, out_features=384, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=384, out_features=1536, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear

In [5]:
def transition(x_t: torch.Tensor, delta_sigma: torch.Tensor) -> torch.Tensor:
    """
    Forward transition kernel:
        exp(σ_t^Δt Q^{tok})(x_t, y)

    Approximates the finite-time forward diffusion probability of moving from token x_t to y
    after a noise increment of Δσ = σ_t^{Δt}.

    Args:
        x_t:          (B, L) integer tensor of current tokens.
        delta_sigma:  scalar tensor representing σ_t^{Δt}.

    Returns:
        trans_probs:  (B, L, V) tensor of categorical probabilities over next tokens.
    """
    # Uniform mixing term from exp(delta_sigma * Q^{tok})
    # with the help of Eq. (3), this translates to:
    base_prob = (1 - torch.exp(-delta_sigma[..., None])) / vocab_size
    trans = torch.ones(*x_t.shape, vocab_size, device=x_t.device) * base_prob

    # Remove the uniform contribution for the current token
    trans = trans.scatter(-1, x_t[..., None], torch.zeros_like(trans))

    # Ensure that probabilities across the vocabulary sum to 1
    diag_fill = 1 - trans.sum(dim=-1, keepdim=True)
    trans = trans.scatter(-1, x_t[..., None], diag_fill)
    return trans


def staggered_score(score, delta_sigma):
    """
    Applies the inverse exponential operator:
        exp(-σ_t^Δt Q^{tok}) s_θ(x_t, t)

    This "staggered" score correction accounts for the finite time-step Δt.

    Args:
        score:        (B, L, V) tensor, model output s_θ(x_t, t)
        delta_sigma:  scalar tensor representing σ_t^{Δt}

    Returns:
        adjusted_score: (B, L, V) tensor, transformed score
    """
    vocab_size = score.shape[-1]
    exp_factor = torch.exp(-delta_sigma)[..., None]  # (B, L, 1)
    correction = ((exp_factor - 1) / (vocab_size * exp_factor)) * score.sum(dim=-1, keepdim=True)
    return correction + score / exp_factor


def sample_categorical(probs: torch.Tensor) -> torch.Tensor:
    """
    Sample from a batch of categorical distributions using the Gumbel-max trick.

    Args:
        probs: (B, L, V) tensor of probabilities that sum to 1 along dim=-1.

    Returns:
        samples: (B, L) tensor of sampled token indices.
    """
    # Add a small epsilon for numerical stability
    eps = 1e-10
    gumbel_noise = -torch.log(-torch.log(torch.rand_like(probs) + eps) + eps)
    return torch.argmax(torch.log(probs + eps) + gumbel_noise, dim=-1)

import random
seed = 42
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
steps = 128
eps = 1e-5
timesteps = torch.linspace(1, eps, steps + 1, device=device)
step_size = (1 - eps) / steps
#x = torch.randint(0, vocab_size, (1, context_length), device=device)
x = torch.tensor([[50, 60, 17, 20, 19, 25, 30, 26, 30,  5, 11, 36, 64, 23, 32, 26, 59, 26,
                   29, 14, 17, 35, 17,  8, 54,  1, 27, 58, 58, 43, 14, 10, 32, 56, 47, 14,
                    4, 38,  7, 56, 44, 29, 21, 10, 27, 31, 12, 13, 20, 26, 50, 20, 58, 43,
                   59, 10,  8, 10, 24,  2, 25, 30, 38, 47, 53, 17, 12, 55, 17, 31, 56, 16,
                   21, 39, 16, 38, 21, 56, 26, 29,  3, 32, 40, 29, 20,  6, 18, 15, 12, 29,
                   54, 48, 24,  9,  8, 49, 37, 10, 61,  3, 20, 41, 40, 47, 19, 35, 20, 35,
                   26, 62, 13,  9, 18, 37, 37, 56,  3, 58, 13, 43,  6,  1, 15, 59, 40, 47,
                   38, 52,  6, 32, 36, 17, 55, 37, 53, 12, 45, 11, 18, 38, 42, 47, 41, 41,
                    6, 36, 17, 19, 14, 14, 31, 31, 22, 58, 39, 24, 17,  9,  3, 26, 63, 60,
                   61, 31, 27, 57, 21,  9, 31, 62,  4, 18, 24, 55, 54, 10, 64, 30, 43, 20,
                    6, 45, 29, 19, 47, 36, 10, 27, 51, 25,  4,  5, 46, 63, 24,  3, 41, 16,
                   30, 42, 35, 58, 56,  0, 10,  4, 55, 63, 53, 48, 57,  2, 19, 14, 63, 35,
                   49, 51, 53, 37, 45, 62, 25, 32,  6, 26, 46, 53,  6, 63, 20, 51, 60, 10,
                   38, 33, 11, 34, 22, 52,  4, 15, 64, 18,  1, 45, 16,  7,  4, 59, 44, 64,
                   34, 17, 29, 45]])

with torch.no_grad():
    for i in range(steps + 1):
        
        t = timesteps[i] * torch.ones(x.shape[0], 1, device=device)
        curr_sigma_bar = noise(t)[0]
        
        
        if i < steps:
            next_sigma_bar = noise(t - step_size)[0]
            
            delta_sigma = curr_sigma_bar - next_sigma_bar

            log_score = model(x, curr_sigma_bar)
            
            score = torch.exp(log_score)
            #print(f"score:{score}")
            #break
            stag_score = staggered_score(score, delta_sigma)
            transitionMatrix = transition(x, delta_sigma)
            #print(f"stag_score:{stag_score}")sv
            #print(f"transitionMatrix:{transitionMatrix}")
            #break
            probs = stag_score * transitionMatrix
            #print(f"probs:{probs}")
            #break
            x = sample_categorical(probs)
            

        else:
            # last denoising step
            # delta_sigma = curr_noise_bar - 0
            delta_sigma = curr_sigma_bar

            log_score = model(x, curr_sigma_bar)
            score = torch.exp(log_score)

            stag_score = staggered_score(score, delta_sigma)
            probs = stag_score * transition(x, delta_sigma)

            x = sample_categorical(probs)

        clear_output(wait=True)
        print(f'Decoded Text at step {i}:', flush=True, end='\n\n')
        print_wrapped(decode(x[0]), end='\n\n', flush=True)
        # time.sleep(0.02)
        

Decoded Text at step 128:

ngers thy delay.
I might thee in thereof is a greater thy sin,
I shall affect thee bid thee there lends there.
Go, Saint Henry, affectment of officers.
WARWICK:
Here I near more thou for me, let in thee day.
Graceive her son he begkit be the duke?

GLOUCES

